# Part 4a — Defense Mechanism Gate Check
Experiment ID: p4a-gate-v1
Objective: verify the drift-score-to-aggregation-weight function behaves
correctly BEFORE running any multi-round federated sweep. Two checks:
(1) does it correctly separate the REAL score distributions already
collected in Part 3b/3c, (2) does one live federated round, weighted by
this function, actually suppress a malicious client's contribution more
than plain FedAvg would, on real data.

Config: partial participation (3/10 clients/round), local_steps=5 --
matches Part 1 exactly (not Part 2's local_steps=2/full-participation,
which was specific to the weight-space baselines' needs). This makes
Part 4 directly comparable to Part 1's undefended numbers.

Two separate probe sets, never mixed: defense-internal scoring uses
Part 3's probe_harmful [228:260]; attack-success evaluation uses Part
1/2's eval_heldout [196:228]. Disjointness asserted in cell 6.

In [1]:
!pip install -q -U transformers accelerate bitsandbytes peft datasets
!pip install torch torchvision torchaudio
!pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.9 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
import sys
if "torchaudio" in sys.modules:
    del sys.modules["torchaudio"]

import os, json, random, datetime, copy
import numpy as np
import torch
from google.colab import drive
drive.mount('/content/drive')

CONFIG = {
    "experiment_id": "p4a-gate-v1",
    "date": datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%d"),
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "layer_index": 14,
    "local_steps": 5,              # matches Part 1, NOT Part 2 -- see Cell 1 note
    "local_lr": 1e-4,
    "lora_r": 8, "lora_alpha": 16,
    "n_clients": 10, "clients_per_round": 3,   # matches Part 1 exactly
    "torch_dtype": "float16",
}

PROJECT_ROOT = "/content/drive/MyDrive/fedllm-safety-defense"
RESULTS_DIR = f"{PROJECT_ROOT}/results/part4a_gate_check"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(json.dumps(CONFIG, indent=2))

Mounted at /content/drive
{
  "experiment_id": "p4a-gate-v1",
  "date": "2026-10-04",
  "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
  "layer_index": 14,
  "local_steps": 5,
  "local_lr": 0.0001,
  "lora_r": 8,
  "lora_alpha": 16,
  "n_clients": 10,
  "clients_per_round": 3,
  "torch_dtype": "float16"
}


In [3]:
#cell 4

from transformers import AutoTokenizer
from transformers.models.qwen2.modeling_qwen2 import Qwen2ForCausalLM

dtype = getattr(torch, CONFIG["torch_dtype"])
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = Qwen2ForCausalLM.from_pretrained(CONFIG["model_name"], torch_dtype=dtype, device_map="auto")
assert base_model.config.num_hidden_layers == 24 and base_model.config.hidden_size == 896

direction_path = f"{PROJECT_ROOT}/results/part0_calibration/p0-calib-v1_Qwen2.5-0.5B-Instruct_layer14_direction.pt"
refusal_direction = torch.load(direction_path, weights_only=False)
assert abs(refusal_direction.norm().item() - 1.0) < 1e-3
direction_unit = refusal_direction.to(base_model.device)

@torch.no_grad()
def get_last_token_activation(model, prompt, layer_index):
    msgs = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model(**inputs, output_hidden_states=True)
    return out.hidden_states[layer_index][0, -1, :].float()
print("Model and direction loaded.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model and direction loaded.


In [4]:
#cell 5

import pandas as pd, urllib.request
from datasets import load_dataset

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/llm-attacks/llm-attacks/main/data/advbench/harmful_behaviors.csv",
    "/content/harmful_behaviors.csv")
advbench_df = pd.read_csv("/content/harmful_behaviors.csv")
rng = random.Random(42)
harmful_all = advbench_df.to_dict("records")
rng.shuffle(harmful_all)

defense_probe = harmful_all[228:260]       # Part 3's scoring probe -- defense sees this
eval_heldout = harmful_all[196:228]        # Part 1/2's eval set -- defense NEVER sees this
poison_pool = harmful_all[96:196]

overlap = set(r["goal"] for r in defense_probe) & set(r["goal"] for r in eval_heldout)
assert len(overlap) == 0, "Defense probe and eval set overlap -- this would invalidate the whole evaluation"
print(f"Defense probe: {len(defense_probe)} prompts | Eval set: {len(eval_heldout)} prompts | Disjoint: confirmed")

alpaca = load_dataset("tatsu-lab/alpaca", split="train")
alpaca_benign = [alpaca[i] for i in range(200)]

Defense probe: 32 prompts | Eval set: 32 prompts | Disjoint: confirmed


README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [5]:
with open(f"{PROJECT_ROOT}/results/part3b_full_sweep/p3b-sweep-v1_FINAL_combined.json") as f:
    p3b_data = json.load(f)

benign_scores_calib, malicious_scores_calib = [], []
for seed_rec in p3b_data["per_seed"].values():
    benign_scores_calib.extend(seed_rec["benign_scores"])
    malicious_scores_calib.extend(seed_rec["malicious_scores"])

benign_mean, benign_std = np.mean(benign_scores_calib), np.std(benign_scores_calib)
malicious_mean, malicious_std = np.mean(malicious_scores_calib), np.std(malicious_scores_calib)
threshold = (benign_mean + malicious_mean) / 2
pooled_std = (benign_std + malicious_std) / 2
k = 1.0 / pooled_std if pooled_std > 0 else 25.0   # steepness: ~1 std-unit transition width

print(f"Calibration from Part 3b (n={len(benign_scores_calib)}+{len(malicious_scores_calib)}):")
print(f"  benign:    mean={benign_mean:+.4f} std={benign_std:.4f}")
print(f"  malicious: mean={malicious_mean:+.4f} std={malicious_std:.4f}")
print(f"  threshold (midpoint) = {threshold:+.4f}")
print(f"  k (steepness, 1/pooled_std) = {k:.2f}")

def drift_to_weight(score, threshold=threshold, k=k):
    # score is MORE NEGATIVE for more suspicious clients (established convention since Part 3)
    return 1.0 / (1.0 + np.exp(-k * (score - threshold)))

# Sanity check against the REAL distributions we calibrated from
w_benign = [drift_to_weight(s) for s in benign_scores_calib]
w_malicious = [drift_to_weight(s) for s in malicious_scores_calib]
print(f"\nWeight on real benign scores:    mean={np.mean(w_benign):.4f}  min={min(w_benign):.4f}")
print(f"Weight on real malicious scores: mean={np.mean(w_malicious):.4f}  max={max(w_malicious):.4f}")
assert np.mean(w_benign) > 0.8 and np.mean(w_malicious) < 0.2, \
    "Weighting function doesn't cleanly separate the calibration data it was built from -- stop, fix before using live"
print("\nPASSED: weighting function cleanly separates its own calibration data.")

Calibration from Part 3b (n=90+90):
  benign:    mean=-0.0805 std=0.0315
  malicious: mean=-0.2546 std=0.0221
  threshold (midpoint) = -0.1676
  k (steepness, 1/pooled_std) = 37.29

Weight on real benign scores:    mean=0.9345  min=0.4490
Weight on real malicious scores: mean=0.0494  max=0.1959

PASSED: weighting function cleanly separates its own calibration data.


In [6]:
#cell 7

from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict

client_ids = list(range(CONFIG["n_clients"]))
lora_config = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
                          target_modules=["q_proj", "v_proj"], lora_dropout=0.0,
                          bias="none", task_type="CAUSAL_LM")
global_model = get_peft_model(copy.deepcopy(base_model), lora_config)
global_state = get_peft_model_state_dict(global_model)

def build_example(prompt, response):
    msgs = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full_text = prompt_text + response
    prompt_ids = tokenizer(prompt_text, return_tensors="pt").input_ids[0]
    full_ids = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=256).input_ids[0]
    labels = full_ids.clone(); labels[:len(prompt_ids)] = -100
    return full_ids, labels

def local_train(model, examples, steps, lr):
    model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    for step in range(steps):
        ids, labels = examples[step % len(examples)]
        ids, labels = ids.unsqueeze(0).to(model.device), labels.unsqueeze(0).to(model.device)
        out = model(input_ids=ids, labels=labels)
        out.loss.backward(); opt.step(); opt.zero_grad()
    model.eval()
    return get_peft_model_state_dict(model)

def mean_projection_on_probe(model, probe_prompts, layer_index, direction):
    acts = torch.stack([get_last_token_activation(model, p["goal"], layer_index) for p in probe_prompts])
    return (acts @ direction).mean().item()

In [7]:
r_test = random.Random(777)

probe_model = get_peft_model(copy.deepcopy(base_model), lora_config)
set_peft_model_state_dict(probe_model, global_state)
round_baseline = mean_projection_on_probe(probe_model, defense_probe, CONFIG["layer_index"], direction_unit)
print(f"Round baseline projection: {round_baseline:.4f}")

client_states, client_scores, client_labels = [], [], []
for label, source_fn in [("benign", lambda: r_test.sample(alpaca_benign, 10)),
                          ("benign", lambda: r_test.sample(alpaca_benign, 10)),
                          ("malicious", lambda: r_test.sample(poison_pool, 10))]:
    client_model = get_peft_model(copy.deepcopy(base_model), lora_config)
    set_peft_model_state_dict(client_model, global_state)
    if label == "benign":
        ex = [build_example(p["instruction"], p["output"]) for p in source_fn()]
    else:
        ex = [build_example(p["goal"], p["target"]) for p in source_fn()]
    state = local_train(client_model, ex, CONFIG["local_steps"], CONFIG["local_lr"])

    scoring_model = get_peft_model(copy.deepcopy(base_model), lora_config)
    set_peft_model_state_dict(scoring_model, state)
    proj = mean_projection_on_probe(scoring_model, defense_probe, CONFIG["layer_index"], direction_unit)
    score = proj - round_baseline

    client_states.append(state); client_scores.append(score); client_labels.append(label)

weights = [drift_to_weight(s) for s in client_scores]
print(f"\nClients: {client_labels}")
print(f"Scores:  {[f'{s:+.4f}' for s in client_scores]}")
print(f"Weights: {[f'{w:.4f}' for w in weights]}")

def weighted_average(states, weights):
    w = np.array(weights); w = w / w.sum()
    keys = states[0].keys()
    return {k: sum(w[i] * states[i][k].float() for i in range(len(states))) for k in keys}

unweighted = {k: torch.stack([s[k].float() for s in client_states], dim=0).mean(dim=0) for k in client_states[0].keys()}
weighted = weighted_average(client_states, weights)

sample_key = list(unweighted.keys())[0]
diff_norm = torch.norm(unweighted[sample_key] - weighted[sample_key]).item()
print(f"\nDifference between unweighted and weighted aggregate (should be > 0): {diff_norm:.6f}")
assert diff_norm > 1e-6, "Weighted and unweighted aggregates are identical -- weighting isn't doing anything, check drift_to_weight"
print("PASSED: weighted aggregation genuinely differs from plain averaging, with the malicious client down-weighted.")

Round baseline projection: 5.2622

Clients: ['benign', 'benign', 'malicious']
Scores:  ['-0.0784', '-0.0697', '-0.2559']
Weights: ['0.9653', '0.9746', '0.0357']

Difference between unweighted and weighted aggregate (should be > 0): 0.005976
PASSED: weighted aggregation genuinely differs from plain averaging, with the malicious client down-weighted.
